# DeepSWE economic lens: descriptive insight audit

## tl;dr

Applying the full economic lens produces a layered result, not one winner. Reported-dollar CPSC
reorders the 13-family capability view and yields a nine-point global frontier composed of GPT-5.6
Luna, Terra, and Sol settings. Within every multi-family provider route, one family supplies the
entire CPSC frontier: Opus 4.8 for Anthropic, Fable 5 for Vertex, and GPT-5.6 variants for OpenAI.

The trial-level decomposition is almost degenerate: failed and successful attempts cost about the
same on average, so reliability-tax share tracks failure rate at 0.995. The more revealing split is
between billed cost and run intensity. Luna max costs 0.39 times Sol max per success but uses 1.80
times the agent steps, 1.32 times the output tokens, and 2.11 times the input tokens. Terra max has
the same reversal. Sol high and Sol xhigh, however, are cheaper than Sol max in dollars, steps, and
tokens while their paired solve-rate differences remain unresolved. Thus CPSC reveals a pricing
frontier; the companion resource metrics reveal whether that frontier reflects less work or merely
a lower price for more work.

## Context & Methods

The question is intentionally neutral: **What patterns appear when realized cost per successful
completion (CPSC), failed-spend decomposition, reliability floors, and failure-price sensitivities
are applied to the DeepSWE v1.1 trials?**

### Key assumptions

- Realized CPSC is benchmark spend divided by verified successes under DeepSWE's observed attempt
  policy. It is not a deployable retry-until-success estimate.
- Cross-provider dollar comparisons use reported `cost_usd` and are descriptive because price,
  cache, and reasoning-token semantics were not harmonized.
- The highest-pass configuration per base model is used only for a readable family-level view.
- Task-difficulty strata are defined after observing the full panel and are descriptive.

In [1]:
from pathlib import Path
import json
import math
import statistics
from random import Random
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
PAPER = ROOT / "paper" / "deepswe-cpsc"
TABLES = PAPER / "generated" / "tables"
REPORT_PATH = Path("/tmp/deepswe-cpsc-v0.1-report.json")
TASKS_PATH = Path("/tmp/deepswe-v1.1-tasks.json")

required = [REPORT_PATH, TASKS_PATH, TABLES / "configuration-results.csv"]
assert all(path.exists() for path in required), required
report = json.loads(REPORT_PATH.read_text())
configurations = pd.read_csv(TABLES / "configuration-results.csv")
best_models = pd.read_csv(TABLES / "best-model-results.csv")
bootstrap = pd.read_csv(TABLES / "bootstrap-intervals.csv")
paired = pd.read_csv(TABLES / "paired-comparisons.csv")
strata = pd.read_csv(TABLES / "task-difficulty-strata.csv")
floors = pd.read_csv(TABLES / "reliability-floor.csv")
failure_prices = pd.read_csv(TABLES / "failure-charge-sensitivity.csv")

print({
    "source_rows": report["primary"]["cohort"]["source_rows"],
    "scored_rows": report["primary"]["cohort"]["scored_rows"],
    "configurations": len(configurations),
    "model_families": configurations["model"].nunique(),
    "tasks": report["bootstrap"]["cluster_count"],
    "bootstrap_replicates": report["bootstrap"]["replicates"],
})

{'source_rows': 18522, 'scored_rows': 18396, 'configurations': 41, 'model_families': 13, 'tasks': 113, 'bootstrap_replicates': 10000}


## Data

### 1. Reconcile the metric and its range

In [2]:
identity = (
    configurations["mean_cost_per_attempt_usd"] / configurations["pass_rate"]
)
identity_error = (identity - configurations["realized_cpsc_usd"]).abs().max()
range_summary = pd.DataFrame({
    "minimum": configurations[["pass_rate", "mean_cost_per_attempt_usd", "realized_cpsc_usd", "realized_reliability_tax_share"]].min(),
    "median": configurations[["pass_rate", "mean_cost_per_attempt_usd", "realized_cpsc_usd", "realized_reliability_tax_share"]].median(),
    "maximum": configurations[["pass_rate", "mean_cost_per_attempt_usd", "realized_cpsc_usd", "realized_reliability_tax_share"]].max(),
})
print(f"Maximum absolute CPSC identity error: {identity_error:.3e}")
display(range_summary.round(4))

Maximum absolute CPSC identity error: 2.842e-14


,minimum,median,maximum
pass_rate,0.0155,0.5177,0.7267
mean_cost_per_attempt_usd,0.0724,3.9199,26.3999
realized_cpsc_usd,1.6612,7.8150,80.6824
realized_reliability_tax_share,0.2756,0.4766,0.9862


### 1b. Data-quality checks

The intended scored grain is one configuration-task attempt. Checks cover uniqueness, task join
coverage, cost validity and imputation, configuration-task cell completeness, official aggregate
reconciliation, and infrastructure exclusions.

In [3]:
derived = pd.read_csv(TABLES / "derived-trials.csv")
task_rows = pd.DataFrame(json.loads(TASKS_PATH.read_text())["rows"])
exclusion_audit = pd.read_csv(TABLES / "infrastructure-exclusion-audit.csv")
exclusion_audit = exclusion_audit.merge(
    configurations[["config", "provider", "model", "reasoning_effort"]],
    on="config",
    how="left",
    validate="one_to_one",
)

cell_sizes = derived.groupby(["config", "task_name"]).size()
quality_summary = {
    "scored_rows": len(derived),
    "duplicate_trial_names": int(derived["trial_name"].duplicated().sum()),
    "task_join_misses": int((~derived["task_name"].isin(task_rows["id"])).sum()),
    "negative_analysis_costs": int((derived["analysis_cost_usd"] < 0).sum()),
    "remaining_missing_analysis_costs": int(derived["analysis_cost_usd"].isna().sum()),
    "imputed_cost_rows": int(derived["cost_imputed"].sum()),
    "complete_four_attempt_cells": int((cell_sizes == 4).sum()),
    "incomplete_cells": int((cell_sizes != 4).sum()),
    "official_aggregate_reconciliation": report["leaderboard_reconciliation"]["all_match"],
    "excluded_infrastructure_rows": report["primary"]["cohort"]["excluded_rows"],
}
print(quality_summary)
display(
    exclusion_audit.sort_values("excluded_attempts", ascending=False)[[
        "provider", "model", "reasoning_effort", "excluded_attempts",
        "included_attempts", "exclusions_as_failures_pass_rate", "included_pass_rate",
    ]].head(10).round(4)
)

{'scored_rows': 18396, 'duplicate_trial_names': 0, 'task_join_misses': 0, 'negative_analysis_costs': 0, 'remaining_missing_analysis_costs': 0, 'imputed_cost_rows': 21, 'complete_four_attempt_cells': 4520, 'incomplete_cells': 111, 'official_aggregate_reconciliation': True, 'excluded_infrastructure_rows': 126}


,provider,model,reasoning_effort,excluded_attempts,included_attempts,exclusions_as_failures_pass_rate,included_pass_rate
7,anthropic,claude-opus-4-8,max,23,429,0.5597,0.5897
0,vertex_ai,claude-fable-5,high,22,430,0.6527,0.6860
1,vertex_ai,claude-fable-5,low,19,433,0.5708,0.5958
2,vertex_ai,claude-fable-5,max,16,436,0.6726,0.6972
3,vertex_ai,claude-fable-5,medium,16,436,0.6305,0.6537
13,anthropic,claude-sonnet-5,max,10,442,0.5265,0.5385
9,anthropic,claude-opus-4-8,xhigh,5,447,0.5376,0.5436
12,anthropic,claude-sonnet-5,low,3,449,0.3031,0.3051
32,openai,gpt-5-6-sol,max,2,450,0.7235,0.7267
19,zai,glm-5-2,max,2,450,0.4358,0.4378


## Results

### 2. Full model-family landscape

One highest-pass configuration per base model, ordered by solve rate. The economic rank is within
this fixed 13-family display panel, not the full 41-configuration menu.

In [4]:
family_view = best_models.copy().sort_values("pass_rate", ascending=False)
family_view["capability_rank_13"] = range(1, len(family_view) + 1)
family_view["economic_rank_13"] = family_view["realized_cpsc_usd"].rank(method="min")
family_view["provider_model"] = family_view["provider"] + " / " + family_view["model"]
display(
    family_view[[
        "capability_rank_13", "economic_rank_13", "provider_model", "reasoning_effort",
        "pass_rate", "mean_cost_per_attempt_usd", "realized_cpsc_usd",
        "realized_reliability_tax_share",
    ]].round(4)
)

,capability_rank_13,economic_rank_13,provider_model,reasoning_effort,pass_rate,mean_cost_per_attempt_usd,realized_cpsc_usd,realized_reliability_tax_share
0,1,7.0,openai / gpt-5-6-sol,max,0.7267,8.3864,11.5410,0.2756
1,2,9.0,vertex_ai / claude-fable-5,xhigh,0.6991,13.4145,19.1879,0.2908
2,3,2.0,openai / gpt-5-6-terra,max,0.6962,4.9458,7.1037,0.3102
3,4,1.0,openai / gpt-5-6-luna,max,0.6719,3.0281,4.5070,0.3177
4,5,5.0,openai / gpt-5-5,xhigh,0.6704,7.2262,10.7797,0.2991
5,6,11.0,anthropic / claude-opus-4-8,max,0.5897,13.2226,22.4209,0.4072
6,7,12.0,anthropic / claude-sonnet-5,max,0.5385,26.3999,49.0283,0.4373
7,8,6.0,openai / gpt-5-4,xhigh,0.5177,5.6525,10.9184,0.4584
8,9,3.0,zai / glm-5-2,max,0.4378,3.9199,8.9541,0.5558
9,10,10.0,vertex_ai / gemini-3-5-flash,medium,0.3739,7.3418,19.6361,0.6601


### 2b. Price-free run intensity

The frozen plan also declared tokens and agent steps per verified success. These distinguish a lower
reported dollar price from lower harness resource use. All displayed family-best configurations
have complete step and token fields. Input-token semantics remain provider-dependent; agent steps
are the most comparable field because every configuration uses the same harness.

In [5]:
resource_totals = derived.groupby("config").agg(
    successes=("passed", "sum"),
    attempts=("passed", "size"),
    complete_resource_rows=("n_agent_steps", "count"),
    input_tokens=("n_input_tokens", "sum"),
    output_tokens=("n_output_tokens", "sum"),
    agent_steps=("n_agent_steps", "sum"),
).reset_index()
for field in ("input_tokens", "output_tokens", "agent_steps"):
    resource_totals[f"{field}_per_success"] = (
        resource_totals[field] / resource_totals["successes"]
    )

resource_family_view = family_view[[
    "config", "provider", "model", "reasoning_effort", "pass_rate", "realized_cpsc_usd"
]].merge(resource_totals, on="config", validate="one_to_one")
resource_family_view["resource_rows_complete"] = (
    resource_family_view["complete_resource_rows"] == resource_family_view["attempts"]
)
display(
    resource_family_view[[
        "provider", "model", "reasoning_effort", "pass_rate", "realized_cpsc_usd",
        "agent_steps_per_success", "output_tokens_per_success", "input_tokens_per_success",
        "resource_rows_complete",
    ]].round(2)
)

,provider,model,reasoning_effort,pass_rate,realized_cpsc_usd,agent_steps_per_success,output_tokens_per_success,input_tokens_per_success,resource_rows_complete
0,openai,gpt-5-6-sol,max,0.73,11.54,84.29,82587.58,1.088209e+07,True
1,vertex_ai,claude-fable-5,xhigh,0.70,19.19,97.84,114934.12,1.048348e+07,True
2,openai,gpt-5-6-terra,max,0.70,7.10,109.06,103325.86,1.325791e+07,True
3,openai,gpt-5-6-luna,max,0.67,4.51,151.34,109246.08,2.298600e+07,True
4,openai,gpt-5-5,xhigh,0.67,10.78,122.35,69060.12,1.249031e+07,True
5,anthropic,claude-opus-4-8,max,0.59,22.42,203.48,228966.75,2.890737e+07,True
6,anthropic,claude-sonnet-5,max,0.54,49.03,498.56,397646.88,1.344864e+08,True
7,openai,gpt-5-4,xhigh,0.52,10.92,136.12,137935.09,1.819381e+07,True
8,zai,glm-5-2,max,0.44,8.95,294.96,178573.04,2.879671e+07,True
9,vertex_ai,gemini-3-5-flash,medium,0.37,19.64,229.25,737582.45,4.575178e+07,True


### 3. Provider-route frontiers

For each provider route, a configuration is nondominated when no other configuration has both an
equal-or-higher solve rate and an equal-or-lower cost. We compare the per-attempt-cost frontier with
the realized-CPSC frontier.

In [6]:
def frontier(group, cost_column):
    keep = []
    records = group.to_dict("records")
    for row in records:
        dominated = any(
            other["pass_rate"] >= row["pass_rate"]
            and other[cost_column] <= row[cost_column]
            and (
                other["pass_rate"] > row["pass_rate"]
                or other[cost_column] < row[cost_column]
            )
            for other in records
        )
        if not dominated:
            keep.append(row)
    return pd.DataFrame(keep).sort_values("pass_rate")

frontier_summary = []
frontier_rows = []
for provider, group in configurations.groupby("provider"):
    attempt_frontier = frontier(group, "mean_cost_per_attempt_usd")
    cpsc_frontier = frontier(group, "realized_cpsc_usd")
    removed = sorted(set(attempt_frontier["config"]) - set(cpsc_frontier["config"]))
    frontier_summary.append({
        "provider": provider,
        "configurations": len(group),
        "attempt_cost_frontier": len(attempt_frontier),
        "cpsc_frontier": len(cpsc_frontier),
        "removed_after_failure_accounting": len(removed),
        "cpsc_frontier_families": ", ".join(sorted(cpsc_frontier["model"].unique())),
    })
    for row in cpsc_frontier.to_dict("records"):
        frontier_rows.append({
            "provider": provider,
            "model": row["model"],
            "effort": row["reasoning_effort"],
            "pass_rate": row["pass_rate"],
            "realized_cpsc_usd": row["realized_cpsc_usd"],
        })

display(pd.DataFrame(frontier_summary))
display(pd.DataFrame(frontier_rows).round(4))

,provider,configurations,attempt_cost_frontier,cpsc_frontier,removed_after_failure_accounting,cpsc_frontier_families
0,anthropic,11,6,5,1,claude-opus-4-8
1,openai,20,13,9,4,"gpt-5-6-luna, gpt-5-6-sol, gpt-5-6-terra"
2,openrouter,1,1,1,0,kimi-k2-7-code
3,vertex_ai,7,4,4,0,claude-fable-5
4,zai,2,2,2,0,glm-5-2


,provider,model,effort,pass_rate,realized_cpsc_usd
0,anthropic,claude-opus-4-8,low,0.4080,5.6212
1,anthropic,claude-opus-4-8,medium,0.4867,7.0756
2,anthropic,claude-opus-4-8,high,0.5177,8.2711
3,anthropic,claude-opus-4-8,xhigh,0.5436,14.7277
4,anthropic,claude-opus-4-8,max,0.5897,22.4209
5,openai,gpt-5-6-terra,medium,0.3511,1.6612
6,openai,gpt-5-6-luna,high,0.4425,1.7581
7,openai,gpt-5-6-terra,high,0.5376,2.1100
8,openai,gpt-5-6-luna,xhigh,0.5686,2.7008
9,openai,gpt-5-6-sol,medium,0.6106,3.0494


### 3b. Dollar and resource frontiers differ

The same nondominance calculation is repeated with steps, output tokens, and input tokens per
verified success. Configurations with incomplete resource fields are excluded from this comparison;
that removes one Sonnet 5 high row and does not affect any displayed frontier.

In [7]:
resource_configurations = configurations.merge(
    resource_totals,
    on="config",
    validate="one_to_one",
)
resource_configurations = resource_configurations[
    resource_configurations["complete_resource_rows"] == resource_configurations["attempts_y"]
].copy()
frontier_metrics = {
    "reported dollar CPSC": "realized_cpsc_usd",
    "agent steps per success": "agent_steps_per_success",
    "output tokens per success": "output_tokens_per_success",
    "input tokens per success": "input_tokens_per_success",
}
resource_frontier_rows = []
resource_frontier_summary = []
for label, metric in frontier_metrics.items():
    metric_frontier = frontier(resource_configurations, metric)
    resource_frontier_summary.append({
        "metric": label,
        "frontier_configurations": len(metric_frontier),
        "frontier_families": ", ".join(sorted(metric_frontier["model"].unique())),
    })
    for row in metric_frontier.to_dict("records"):
        resource_frontier_rows.append({
            "metric": label,
            "configuration": row["config"],
            "pass_rate": row["pass_rate"],
            "value": row[metric],
        })
display(pd.DataFrame(resource_frontier_summary))
display(pd.DataFrame(resource_frontier_rows).round(3))

,metric,frontier_configurations,frontier_families
0,reported dollar CPSC,9,"gpt-5-6-luna, gpt-5-6-sol, gpt-5-6-terra"
1,agent steps per success,4,gpt-5-6-sol
2,output tokens per success,5,gpt-5-6-sol
3,input tokens per success,5,gpt-5-6-sol


,metric,configuration,pass_rate,value
0,reported dollar CPSC,mini_swe_agent_gpt_5_6_terra_medium,0.351,1.661000e+00
1,reported dollar CPSC,mini_swe_agent_gpt_5_6_luna_high,0.442,1.758000e+00
2,reported dollar CPSC,mini_swe_agent_gpt_5_6_terra_high,0.538,2.110000e+00
3,reported dollar CPSC,mini_swe_agent_gpt_5_6_luna_xhigh,0.569,2.701000e+00
4,reported dollar CPSC,mini_swe_agent_gpt_5_6_sol_medium,0.611,3.049000e+00
5,reported dollar CPSC,mini_swe_agent_gpt_5_6_luna_max,0.672,4.507000e+00
6,reported dollar CPSC,mini_swe_agent_gpt_5_6_sol_high,0.694,5.000000e+00
7,reported dollar CPSC,mini_swe_agent_gpt_5_6_sol_xhigh,0.707,6.650000e+00
8,reported dollar CPSC,mini_swe_agent_gpt_5_6_sol_max,0.727,1.154100e+01
9,agent steps per success,mini_swe_agent_gpt_5_6_sol_medium,0.611,5.062700e+01


### 4. Effort scaling within model families

This view asks where CPSC is minimized within each family and whether increasing effort ever raises
solve rate while lowering realized CPSC.

In [8]:
effort_order = {"low": 0, "medium": 1, "high": 2, "xhigh": 3, "max": 4, "default": 2}
family_effort_summary = []
strict_improvements = []
for model, group in configurations.groupby("model"):
    if len(group) < 2:
        continue
    ordered = group.assign(
        effort_order=group["reasoning_effort"].map(effort_order).fillna(99)
    ).sort_values("effort_order")
    minimum = ordered.loc[ordered["realized_cpsc_usd"].idxmin()]
    maximum_pass = ordered.loc[ordered["pass_rate"].idxmax()]
    family_effort_summary.append({
        "model": model,
        "effort_levels": len(ordered),
        "minimum_cpsc_effort": minimum["reasoning_effort"],
        "minimum_cpsc_usd": minimum["realized_cpsc_usd"],
        "pass_rate_at_minimum": minimum["pass_rate"],
        "maximum_pass_effort": maximum_pass["reasoning_effort"],
        "maximum_pass_rate": maximum_pass["pass_rate"],
        "cpsc_at_maximum_pass": maximum_pass["realized_cpsc_usd"],
    })
    records = ordered.to_dict("records")
    for lower, higher in zip(records, records[1:]):
        if (
            higher["pass_rate"] > lower["pass_rate"]
            and higher["realized_cpsc_usd"] < lower["realized_cpsc_usd"]
        ):
            strict_improvements.append({
                "model": model,
                "transition": f"{lower['reasoning_effort']} -> {higher['reasoning_effort']}",
                "pass_rate_before": lower["pass_rate"],
                "pass_rate_after": higher["pass_rate"],
                "cpsc_before": lower["realized_cpsc_usd"],
                "cpsc_after": higher["realized_cpsc_usd"],
            })

display(pd.DataFrame(family_effort_summary).round(4))
display(pd.DataFrame(strict_improvements).round(4))

,model,effort_levels,minimum_cpsc_effort,minimum_cpsc_usd,pass_rate_at_minimum,maximum_pass_effort,maximum_pass_rate,cpsc_at_maximum_pass
0,claude-fable-5,5,low,6.3068,0.5958,xhigh,0.6991,19.1879
1,claude-opus-4-8,5,low,5.6212,0.4080,max,0.5897,22.4209
2,claude-sonnet-5,5,low,7.1662,0.3051,max,0.5385,49.0283
3,glm-5-2,2,high,7.8150,0.3628,max,0.4378,8.9541
4,gpt-5-5,4,low,4.4467,0.2699,xhigh,0.6704,10.7797
5,gpt-5-6-luna,5,high,1.7581,0.4425,max,0.6719,4.5070
6,gpt-5-6-sol,5,low,2.3686,0.4535,max,0.7267,11.5410
7,gpt-5-6-terra,5,medium,1.6612,0.3511,max,0.6962,7.1037


,model,transition,pass_rate_before,pass_rate_after,cpsc_before,cpsc_after
0,gpt-5-6-luna,low -> medium,0.0155,0.1128,4.6754,1.9171
1,gpt-5-6-luna,medium -> high,0.1128,0.4425,1.9171,1.7581
2,gpt-5-6-terra,low -> medium,0.2405,0.3511,1.7783,1.6612


#### Uncertainty for the three apparent higher-effort improvements

A point estimate that raises solve rate and lowers CPSC is only descriptive until the paired
task-bootstrap interval is checked.

In [9]:
def normalized_pair(lower_config, higher_config):
    match = paired[
        ((paired["config_a"] == lower_config) & (paired["config_b"] == higher_config))
        | ((paired["config_a"] == higher_config) & (paired["config_b"] == lower_config))
    ].iloc[0]
    higher_is_a = match["config_a"] == higher_config
    if higher_is_a:
        pass_low = match["pass_rate_difference_ci_low"]
        pass_high = match["pass_rate_difference_ci_high"]
        ratio = math.exp(match["point_log_cpsc_ratio"])
        ratio_low = math.exp(match["log_cpsc_ratio_ci_low"])
        ratio_high = math.exp(match["log_cpsc_ratio_ci_high"])
        cheaper = match["a_cheaper_replicates"]
    else:
        pass_low = -match["pass_rate_difference_ci_high"]
        pass_high = -match["pass_rate_difference_ci_low"]
        ratio = math.exp(-match["point_log_cpsc_ratio"])
        ratio_low = math.exp(-match["log_cpsc_ratio_ci_high"])
        ratio_high = math.exp(-match["log_cpsc_ratio_ci_low"])
        cheaper = (
            match["defined_replicates"]
            - match["a_cheaper_replicates"]
            - match["tied_cpsc_replicates"]
        )
    return {
        "transition": f"{lower_config} -> {higher_config}",
        "pass_difference_ci_low": pass_low,
        "pass_difference_ci_high": pass_high,
        "higher_effort_cpsc_ratio": ratio,
        "ratio_ci_low": ratio_low,
        "ratio_ci_high": ratio_high,
        "higher_effort_cheaper_replicates": int(cheaper),
        "defined_replicates": int(match["defined_replicates"]),
    }

improvement_pairs = [
    ("mini_swe_agent_gpt_5_6_luna_low", "mini_swe_agent_gpt_5_6_luna_medium"),
    ("mini_swe_agent_gpt_5_6_luna_medium", "mini_swe_agent_gpt_5_6_luna_high"),
    ("mini_swe_agent_gpt_5_6_terra_low", "mini_swe_agent_gpt_5_6_terra_medium"),
]
display(pd.DataFrame(normalized_pair(*pair) for pair in improvement_pairs).round(4))

,transition,pass_difference_ci_low,pass_difference_ci_high,higher_effort_cpsc_ratio,ratio_ci_low,ratio_ci_high,higher_effort_cheaper_replicates,defined_replicates
0,mini_swe_agent_gpt_5_6_luna_low -> mini_swe_ag...,0.0619,0.1350,0.4100,0.1052,0.7912,9911,9937
1,mini_swe_agent_gpt_5_6_luna_medium -> mini_swe...,0.2721,0.3872,0.9170,0.6382,1.2150,7240,10000
2,mini_swe_agent_gpt_5_6_terra_low -> mini_swe_a...,0.0570,0.1641,0.9341,0.7559,1.1282,7571,10000


### 5. Failure-spend decomposition

We measure how closely the reliability-tax share tracks the failure rate and compare mean failed
attempt spend with mean successful attempt spend.

In [10]:
tax_correlation = configurations["realized_reliability_tax_share"].corr(
    1 - configurations["pass_rate"]
)
failed_success_ratio = (
    configurations["conditional_failed_spend_usd"]
    / configurations["conditional_successful_spend_usd"]
)
print({
    "correlation_tax_share_with_failure_rate": round(tax_correlation, 4),
    "failed_to_successful_attempt_spend_min": round(failed_success_ratio.min(), 4),
    "failed_to_successful_attempt_spend_median": round(failed_success_ratio.median(), 4),
    "failed_to_successful_attempt_spend_max": round(failed_success_ratio.max(), 4),
    "configurations_where_failure_costs_more_on_average": int((failed_success_ratio > 1).sum()),
})

{'correlation_tax_share_with_failure_rate': np.float64(0.995), 'failed_to_successful_attempt_spend_min': np.float64(0.7196), 'failed_to_successful_attempt_spend_median': np.float64(1.0155), 'failed_to_successful_attempt_spend_max': np.float64(1.2306), 'configurations_where_failure_costs_more_on_average': 22}


### 6. Post-hoc task-difficulty strata

Rare, contested, and common are defined by how many of the 41 configurations solve a task at least
once. This is useful for description but unavailable before running the panel.

In [11]:
stratum_summary = []
for stratum, group in strata.groupby("stratum", sort=False):
    defined = group.dropna(subset=["realized_cpsc_usd"])
    cheapest = defined.loc[defined["realized_cpsc_usd"].idxmin()]
    leader = group.loc[group["pass_rate"].idxmax()]
    stratum_summary.append({
        "stratum": stratum,
        "tasks": int(group["tasks_in_stratum"].iloc[0]),
        "cpsc_minimum_config": cheapest["config"],
        "cpsc_minimum_pass_rate": cheapest["pass_rate"],
        "cpsc_minimum_usd": cheapest["realized_cpsc_usd"],
        "capability_leader": leader["config"],
        "leader_pass_rate": leader["pass_rate"],
        "leader_cpsc_usd": leader["realized_cpsc_usd"],
    })
display(pd.DataFrame(stratum_summary).round(4))

,stratum,tasks,cpsc_minimum_config,cpsc_minimum_pass_rate,cpsc_minimum_usd,capability_leader,leader_pass_rate,leader_cpsc_usd
0,rare,17,mini_swe_agent_gpt_5_6_luna_max,0.3676,11.0698,mini_swe_agent_claude_fable_5_max,0.3692,68.7577
1,contested,31,mini_swe_agent_gpt_5_6_luna_high,0.2661,3.0564,mini_swe_agent_gpt_5_6_sol_max,0.5984,15.7626
2,common,65,mini_swe_agent_gpt_5_6_terra_medium,0.5290,1.0656,mini_swe_agent_gpt_5_6_sol_max,0.9115,8.3496


### 7. Near-top capability configurations

We compare Sol max with every configuration whose paired task-bootstrap solve-rate difference has a
95% interval containing zero. This is unresolved difference, not evidence of equivalence.

In [12]:
by_config = configurations.set_index("config")
sol_max = "mini_swe_agent_gpt_5_6_sol_max"
near_top = []
for row in paired.to_dict("records"):
    if sol_max not in {row["config_a"], row["config_b"]}:
        continue
    other = row["config_b"] if row["config_a"] == sol_max else row["config_a"]
    if row["config_a"] == other:
        pass_low = row["pass_rate_difference_ci_low"]
        pass_high = row["pass_rate_difference_ci_high"]
        ratio = math.exp(row["point_log_cpsc_ratio"])
        ratio_low = math.exp(row["log_cpsc_ratio_ci_low"])
        ratio_high = math.exp(row["log_cpsc_ratio_ci_high"])
        cheaper = row["a_cheaper_replicates"]
    else:
        pass_low = -row["pass_rate_difference_ci_high"]
        pass_high = -row["pass_rate_difference_ci_low"]
        ratio = math.exp(-row["point_log_cpsc_ratio"])
        ratio_low = math.exp(-row["log_cpsc_ratio_ci_high"])
        ratio_high = math.exp(-row["log_cpsc_ratio_ci_low"])
        cheaper = row["defined_replicates"] - row["a_cheaper_replicates"] - row["tied_cpsc_replicates"]
    if pass_low <= 0 <= pass_high:
        other_row = by_config.loc[other]
        near_top.append({
            "provider": other_row["provider"],
            "configuration": other,
            "pass_rate": other_row["pass_rate"],
            "pass_difference_ci_low": pass_low,
            "pass_difference_ci_high": pass_high,
            "cpsc_usd": other_row["realized_cpsc_usd"],
            "cpsc_ratio_vs_sol_max": ratio,
            "ratio_ci_low": ratio_low,
            "ratio_ci_high": ratio_high,
            "cheaper_replicates": int(cheaper),
            "defined_replicates": int(row["defined_replicates"]),
        })
display(pd.DataFrame(near_top).sort_values("cpsc_usd").round(4))

,provider,configuration,pass_rate,pass_difference_ci_low,pass_difference_ci_high,cpsc_usd,cpsc_ratio_vs_sol_max,ratio_ci_low,ratio_ci_high,cheaper_replicates,defined_replicates
3,openai,mini_swe_agent_gpt_5_6_luna_max,0.6719,-0.1215,0.0115,4.5070,0.3905,0.3455,0.4393,10000,10000
4,openai,mini_swe_agent_gpt_5_6_sol_high,0.6940,-0.0772,0.0112,4.9997,0.4332,0.3999,0.4692,10000,10000
5,openai,mini_swe_agent_gpt_5_6_sol_xhigh,0.7073,-0.0563,0.0167,6.6500,0.5762,0.5362,0.6213,10000,10000
6,openai,mini_swe_agent_gpt_5_6_terra_max,0.6962,-0.0808,0.0188,7.1037,0.6155,0.5625,0.6745,10000,10000
0,vertex_ai,mini_swe_agent_claude_fable_5_high,0.6860,-0.1098,0.0293,13.3776,1.1591,1.0220,1.3155,116,10000
2,vertex_ai,mini_swe_agent_claude_fable_5_xhigh,0.6991,-0.0998,0.0433,19.1879,1.6626,1.4724,1.8832,0,10000
1,vertex_ai,mini_swe_agent_claude_fable_5_max,0.6972,-0.1035,0.0443,31.0287,2.6886,2.3726,3.0621,0,10000


#### Price-free ratios within the near-top set

Ratios compare each configuration with Sol max. Intervals use the same 10,000 paired task-cluster
bootstrap draws and seed as the dollar analysis. Values below one use fewer resources per verified
success.

In [13]:
near_top_configs = [sol_max] + [row["configuration"] for row in near_top]
near_top_trials = derived[derived["config"].isin(near_top_configs)].copy()
assert near_top_trials[["n_agent_steps", "n_output_tokens", "n_input_tokens"]].notna().all().all()
task_names = sorted(derived["task_name"].unique())

random = Random(20260714)
task_counts = np.zeros((10000, len(task_names)), dtype=np.int16)
for replicate in range(10000):
    for _ in task_names:
        task_counts[replicate, random.randrange(len(task_names))] += 1

resource_fields = ["n_agent_steps", "n_output_tokens", "n_input_tokens"]
bootstrap_intensity = {}
for config in near_top_configs:
    group = near_top_trials[near_top_trials["config"] == config]
    successes_by_task = (
        group.groupby("task_name")["passed"].sum().reindex(task_names, fill_value=0).to_numpy(float)
    )
    success_denominator = task_counts @ successes_by_task
    bootstrap_intensity[config] = {}
    for field in resource_fields:
        resource_by_task = (
            group.groupby("task_name")[field].sum().reindex(task_names, fill_value=0).to_numpy(float)
        )
        bootstrap_intensity[config][field] = (
            task_counts @ resource_by_task
        ) / success_denominator

resource_ratio_rows = []
for row in sorted(near_top, key=lambda item: item["cpsc_usd"]):
    config = row["configuration"]
    result = {
        "provider": by_config.loc[config, "provider"],
        "configuration": config,
        "dollar_cpsc_ratio": row["cpsc_ratio_vs_sol_max"],
    }
    for field in resource_fields:
        ratios = bootstrap_intensity[config][field] / bootstrap_intensity[sol_max][field]
        short = field.removeprefix("n_").removesuffix("_tokens")
        result[f"{short}_ratio"] = float(np.median(ratios))
        result[f"{short}_ci_low"] = float(np.quantile(ratios, 0.025))
        result[f"{short}_ci_high"] = float(np.quantile(ratios, 0.975))
    resource_ratio_rows.append(result)

display(pd.DataFrame(resource_ratio_rows).round(3))

,provider,configuration,dollar_cpsc_ratio,agent_steps_ratio,agent_steps_ci_low,agent_steps_ci_high,output_ratio,output_ci_low,output_ci_high,input_ratio,input_ci_low,input_ci_high
0,openai,mini_swe_agent_gpt_5_6_luna_max,0.391,1.796,1.619,1.994,1.323,1.198,1.460,2.113,1.873,2.378
1,openai,mini_swe_agent_gpt_5_6_sol_high,0.433,0.631,0.588,0.677,0.496,0.464,0.531,0.359,0.331,0.389
2,openai,mini_swe_agent_gpt_5_6_sol_xhigh,0.576,0.738,0.693,0.787,0.698,0.660,0.739,0.554,0.516,0.595
3,openai,mini_swe_agent_gpt_5_6_terra_max,0.616,1.294,1.192,1.408,1.251,1.163,1.348,1.218,1.107,1.345
4,vertex_ai,mini_swe_agent_claude_fable_5_high,1.159,1.015,0.913,1.132,1.011,0.904,1.129,0.643,0.553,0.760
5,vertex_ai,mini_swe_agent_claude_fable_5_xhigh,1.663,1.160,1.041,1.297,1.391,1.243,1.558,0.962,0.845,1.103
6,vertex_ai,mini_swe_agent_claude_fable_5_max,2.689,1.505,1.340,1.694,2.060,1.847,2.300,1.667,1.465,1.897


### 8. Reliability and failure-price sensitivities

In [14]:
floor_changes = floors.loc[
    floors["minimum_cpsc_config"].ne(floors["minimum_cpsc_config"].shift())
].copy()
display(floor_changes.round(4))

failure_price_winners = []
for multiplier, group in failure_prices.groupby("multiplier"):
    winner = group.loc[group["reference_budget_cpsc_usd"].idxmin()]
    failure_price_winners.append({
        "multiplier": multiplier,
        "winner": winner["config"],
        "pass_rate": winner["pass_rate"],
        "reference_budget_cpsc_usd": winner["reference_budget_cpsc_usd"],
    })
display(pd.DataFrame(failure_price_winners).round(4))

,eligible_configurations,minimum_cpsc_config,minimum_cpsc_usd,minimum_pass_rate,observed_pass_rate
0,41,mini_swe_agent_gpt_5_6_terra_medium,1.6612,0.00,0.3511
8,29,mini_swe_agent_gpt_5_6_luna_high,1.7581,0.40,0.4425
9,26,mini_swe_agent_gpt_5_6_terra_high,2.1100,0.45,0.5376
11,16,mini_swe_agent_gpt_5_6_luna_xhigh,2.7008,0.55,0.5686
12,13,mini_swe_agent_gpt_5_6_sol_medium,3.0494,0.60,0.6106
13,10,mini_swe_agent_gpt_5_6_luna_max,4.5070,0.65,0.6719
14,2,mini_swe_agent_gpt_5_6_sol_xhigh,6.6500,0.70,0.7073
15,0,NaN,NaN,0.75,NaN


,multiplier,winner,pass_rate,reference_budget_cpsc_usd
0,0.5,mini_swe_agent_gpt_5_6_sol_medium,0.6106,4.4603
1,1.0,mini_swe_agent_gpt_5_6_luna_max,0.6719,6.8628
2,2.0,mini_swe_agent_gpt_5_6_luna_max,0.6719,10.6505


#### Selection stability at the frontier change points

In [15]:
floor_bootstrap = pd.read_csv(TABLES / "reliability-floor-bootstrap.csv")
change_points = set(floor_changes["minimum_pass_rate"].dropna())
policy_view = floor_bootstrap[
    floor_bootstrap["minimum_pass_rate"].isin(change_points)
][[
    "minimum_pass_rate", "point_estimate_winner", "point_winner_selection_share_all_replicates",
    "most_selected_config", "most_selected_share_all_replicates",
    "no_eligible_configuration_replicates",
]]
display(policy_view.round(4))

,minimum_pass_rate,point_estimate_winner,point_winner_selection_share_all_replicates,most_selected_config,most_selected_share_all_replicates,no_eligible_configuration_replicates
0,0.00,mini_swe_agent_gpt_5_6_terra_medium,0.5345,mini_swe_agent_gpt_5_6_terra_medium,0.5345,0
8,0.40,mini_swe_agent_gpt_5_6_luna_high,0.8345,mini_swe_agent_gpt_5_6_luna_high,0.8345,0
9,0.45,mini_swe_agent_gpt_5_6_terra_high,0.5689,mini_swe_agent_gpt_5_6_terra_high,0.5689,0
11,0.55,mini_swe_agent_gpt_5_6_luna_xhigh,0.3783,mini_swe_agent_gpt_5_6_luna_xhigh,0.3783,0
12,0.60,mini_swe_agent_gpt_5_6_sol_medium,0.4261,mini_swe_agent_gpt_5_6_sol_medium,0.4261,1
13,0.65,mini_swe_agent_gpt_5_6_luna_max,0.5732,mini_swe_agent_gpt_5_6_luna_max,0.5732,26
14,0.70,mini_swe_agent_gpt_5_6_sol_xhigh,0.1636,mini_swe_agent_gpt_5_6_sol_high,0.2985,1171
15,0.75,NaN,NaN,mini_swe_agent_gpt_5_6_sol_max,0.1326,6707


## Takeaways

1. **CPSC changes the descriptive view, but its meaning is specific.** It measures reported
   benchmark dollars per verified success. It is not a universal resource-efficiency score.
2. **The broad provider result is a set of efficient families, not a Luna/Sol story.** Opus 4.8,
   Fable 5, and GPT-5.6 variants supply the provider-route CPSC frontiers. Open-weight GLM and Kimi
   remain descriptive standalone routes because there is no within-route family comparison and no
   common-price reconstruction.
3. **Failure accounting mostly removes false economies at very low reliability.** Moving from
   per-attempt cost to CPSC removes five low-reliability configurations from provider frontiers.
   Only Luna low to medium has a paired interval resolving both higher pass and lower CPSC, and
   both settings remain low reliability.
4. **Trial-level data add uncertainty and outcome-conditioned behavior, not the aggregate ratio.**
   Seven configurations have solve-rate differences from Sol max that are unresolved under task
   resampling, while their dollar CPSC differences are strongly resolved. Failed and successful
   attempts have similar spend, and high reliability-floor choices are selection-unstable.
5. **Task difficulty changes the economical operating point.** The post-hoc CPSC minimum moves
   from Terra medium on common tasks to Luna high on contested tasks and Luna max on rare tasks.
   This is descriptive because the strata use outcomes from the full panel.
6. **A coherent research question emerges without forcing a positive result:** what do
   failure-aware dollar, token, and step economics reveal beyond a capability leaderboard on a
   fixed-attempt frontier benchmark? DeepSWE's answer is that price, resource intensity, and
   reliability produce different frontiers, while the failure-tax decomposition itself adds little
   because attempts consume similar budgets regardless of outcome.